# 예제 05. CNN에서 만나는 오류
빅데이터프로그래밍 · 8주차

아래 셀들은 **일부러 오류가 나도록** 만들어져 있습니다.
CNN 실습에서 학생들이 실제로 만나는 것들입니다.


In [ ]:
import torch
import torch.nn as nn

x = torch.randn(4, 1, 28, 28)
print("입력:", tuple(x.shape))


## 1. Flatten 뒤 Linear 크기 오류 — 가장 많이 만납니다


In [ ]:
bad = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(1000, 10),        # 1568 이어야 함
)
try:
    bad(x)
except RuntimeError as err:
    print("RuntimeError:", err)


In [ ]:
# 해결: 실제 크기를 먼저 확인한다
features = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
)
with torch.no_grad():
    print("실제 Flatten 크기:", features(torch.zeros(1,1,28,28)).numel())


## 2. 채널 수 불일치 — 앞 층의 out_channels ≠ 뒤 층의 in_channels


In [ ]:
bad2 = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(8, 32, 3, padding=1),      # 16 이어야 함
)
try:
    bad2(x)
except RuntimeError as err:
    print("RuntimeError:", err)


## 3. 3차원 입력 — batch 차원이 없다
`Conv2d` 는 (batch, 채널, 높이, 너비) 4차원을 받습니다.


In [ ]:
one_image = torch.randn(1, 28, 28)      # 채널, 높이, 너비만
conv = nn.Conv2d(1, 16, 3)
try:
    conv(one_image)
except RuntimeError as err:
    print("RuntimeError:", err)


In [ ]:
# 해결: unsqueeze(0) 으로 batch 차원 추가 — 4주차에서 배운 것
print("해결:", tuple(conv(one_image.unsqueeze(0)).shape))


## 4. 채널 축이 뒤에 있다
matplotlib는 (높이, 너비, 채널)을, PyTorch는 (채널, 높이, 너비)를 씁니다.


In [ ]:
hwc = torch.randn(1, 28, 28, 3)         # 뒤에 채널이 온 컬러 이미지
conv_rgb = nn.Conv2d(3, 16, 3)
try:
    conv_rgb(hwc)
except RuntimeError as err:
    print("RuntimeError:", err)


In [ ]:
# 해결: permute 로 축 순서 바꾸기 — 4주차에서 배운 것
chw = hwc.permute(0, 3, 1, 2)
print("바꾼 뒤:", tuple(chw.shape))
print("해결:", tuple(conv_rgb(chw).shape))


## 5. 풀링을 너무 많이 해서 크기가 0이 됐다


In [ ]:
too_much = nn.Sequential(*[nn.MaxPool2d(2) for _ in range(6)])
try:
    print(tuple(too_much(x).shape))
except RuntimeError as err:
    print("RuntimeError:", err)

# 28 → 14 → 7 → 3 → 1 → 0
print("\n28을 2로 계속 나누면:", [28 // (2**i) for i in range(6)])


## 오류 대응 요약

| 증상 | 원인 | 해결 |
| --- | --- | --- |
| mat1 and mat2 shapes cannot be multiplied | Flatten 뒤 Linear 크기 틀림 | 더미 입력으로 실제 크기 확인 |
| expected input to have N channels | 채널 수 불일치 | 앞 층 out = 뒤 층 in |
| Expected 4-dimensional input | batch 차원 없음 | `unsqueeze(0)` |
| Given groups=1, weight of size ... | 채널 축 위치 틀림 | `permute(0, 3, 1, 2)` |
| Output size is too small | 풀링 과다 | 층 수 줄이기 |

## 직접 해보기
아래 셀에는 오류가 두 개 있습니다. 메시지를 읽고 고치세요.


In [ ]:
m = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(4, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16 * 14 * 14, 10),
)
print(m(torch.randn(2, 1, 28, 28)).shape)
